# Normalization Debug Notebook

Steps through the exact code used in `process()` for loading and normalizing static attributes,
then verifies the result matches what was actually saved in the processed base graphs.

Each cell is copied verbatim from the production code with check/display statements added between steps.

In [10]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import RobustScaler

from src.data.loader import LeakDBLoader
from src.data.graph_builder import GraphBuilder
from src.utils.config import Config

# data_dir must point one level up since the notebook lives in experiments/
config = Config(data_dir='../data')
loader = LeakDBLoader(config.raw_data_dir)
topology = loader.load_topology()

n_junctions = sum(1 for t in topology['nodes'].values() if t['type'] == 'junction')
n_reservoirs = sum(1 for t in topology['nodes'].values() if t['type'] == 'reservoir')
n_pipes = len(topology['pipes'])

print(f"Scenarios : {config.scenarios}")
print(f"Junctions : {n_junctions}")
print(f"Reservoirs: {n_reservoirs}")
print(f"Pipes     : {n_pipes}")

Scenarios : [1]
Junctions : 31
Reservoirs: 1
Pipes     : 34


## Step 1 — Load raw attributes
Exact copy of the `load_attributes_as_df` call in `process()`.

In [11]:
# --- copied from process() Step 4 ---
attrs_dfs = loader.load_attributes_as_df(config.scenarios)
junc_df = attrs_dfs['junctions']  # index: (node_id, scenario), col: base_demand
pipe_df = attrs_dfs['pipes']      # index: (pipe_id, scenario), cols: length, diameter, roughness

print(f"junc_df shape: {junc_df.shape}  —  columns: {list(junc_df.columns)}")
print(f"pipe_df shape: {pipe_df.shape}  —  columns: {list(pipe_df.columns)}")

print("\nNaN in junc_df:", junc_df.isna().sum().to_dict())
print("NaN in pipe_df:", pipe_df.isna().sum().to_dict())

print("\n--- Raw junction attributes ---")
display(junc_df.describe())

print("\n--- Raw pipe attributes ---")
display(pipe_df.describe())

display(junc_df)
display(pipe_df)

junc_df shape: (31, 1)  —  columns: ['base_demand']
pipe_df shape: (34, 3)  —  columns: ['length', 'diameter', 'roughness']

NaN in junc_df: {'base_demand': 0}
NaN in pipe_df: {'length': 0, 'diameter': 0, 'roughness': 0}

--- Raw junction attributes ---


,base_demand
count,31.000000
mean,0.050926
std,0.029375
min,0.004444
25%,0.028066
50%,0.046981
75%,0.070116
max,0.118648



--- Raw pipe attributes ---


,length,diameter,roughness
count,34.000000,34.000000,34.000000
mean,1164.826544,0.658680,129.977385
std,761.421692,0.287192,7.336033
min,101.822749,0.274364,117.006376
25%,775.468690,0.413817,124.688268
50%,956.441918,0.596033,130.968590
75%,1449.669389,0.950945,134.787971
max,3352.516158,1.090574,142.519651


,,base_demand
node_id,scenario,
10,1,0.043271
11,1,0.037697
12,1,0.044148
13,1,0.081994
14,1,0.050555
15,1,0.018951
16,1,0.023779
17,1,0.062472
18,1,0.118648


,,length,diameter,roughness
pipe_id,scenario,,,
1,1,101.822749,1.032579,132.031542
10,1,987.517402,0.781011,117.006376
11,1,1238.726940,0.801789,134.917193
12,1,3352.516158,0.610025,129.311862
13,1,801.163063,0.432517,142.519651
14,1,522.604572,0.410238,137.877539
15,1,550.519997,0.293991,135.751098
16,1,2858.392928,0.369400,131.599908
17,1,1773.933313,0.514261,137.012928


## Step 2 — Fit and apply RobustScaler
Exact copy of the scaler block in `process()`.

In [12]:
# --- copied from process() Step 4 ---
junc_scaler = RobustScaler()
pipe_scaler  = RobustScaler()

junc_norm = pd.DataFrame(
    junc_scaler.fit_transform(junc_df),
    index=junc_df.index, columns=junc_df.columns,
)
pipe_norm = pd.DataFrame(
    pipe_scaler.fit_transform(pipe_df),
    index=pipe_df.index, columns=pipe_df.columns,
)

print("=== Junction scaler parameters ===")
for col, c, s in zip(junc_df.columns, junc_scaler.center_, junc_scaler.scale_):
    print(f"  {col:15s}: center={c:.6f}  scale={s:.6f}")

print("\n=== Pipe scaler parameters ===")
for col, c, s in zip(pipe_df.columns, pipe_scaler.center_, pipe_scaler.scale_):
    print(f"  {col:15s}: center={c:.6f}  scale={s:.6f}")

print("\nNaN in junc_norm:", junc_norm.isna().sum().to_dict())
print("NaN in pipe_norm:", pipe_norm.isna().sum().to_dict())

print("\n--- Normalized junction attributes ---")
display(junc_norm.describe())

print("\n--- Normalized pipe attributes ---")
display(pipe_norm.describe())

=== Junction scaler parameters ===
  base_demand    : center=0.046981  scale=0.042050

=== Pipe scaler parameters ===
  length         : center=956.441918  scale=674.200700
  diameter       : center=0.596033  scale=0.537128
  roughness      : center=130.968590  scale=10.099703

NaN in junc_norm: {'base_demand': 0}
NaN in pipe_norm: {'length': 0, 'diameter': 0, 'roughness': 0}

--- Normalized junction attributes ---


,base_demand
count,31.000000
mean,0.093818
std,0.698559
min,-1.011572
25%,-0.449826
50%,0.000000
75%,0.550174
max,1.704304



--- Normalized pipe attributes ---


,length,diameter,roughness
count,34.000000,3.400000e+01,34.000000
mean,0.309084,1.166335e-01,-0.098142
std,1.129369,5.346816e-01,0.726361
min,-1.267604,-5.988687e-01,-1.382438
25%,-0.268426,-3.392415e-01,-0.621832
50%,0.000000,1.040834e-16,0.000000
75%,0.731574,6.607585e-01,0.378168
max,3.553948,9.207141e-01,1.143703


## Step 3 — Slice per scenario with `xs()`
Exact copy of the per-scenario slice in `process()`.

In [13]:
scenario = config.scenarios[0]

# --- copied from process() Step 5 scenario loop ---
junc_s = junc_norm.xs(scenario, level='scenario')
pipe_s = pipe_norm.xs(scenario, level='scenario')

print(f"=== junc_s (scenario {scenario}) ===")
print(f"Index name : {junc_s.index.name}")
print(f"Index dtype: {junc_s.index.dtype}")
print(f"Shape      : {junc_s.shape}")
display(junc_s)

print(f"\n=== pipe_s (scenario {scenario}) ===")
print(f"Index name : {pipe_s.index.name}")
print(f"Index dtype: {pipe_s.index.dtype}")
print(f"Shape      : {pipe_s.shape}")
display(pipe_s)

=== junc_s (scenario 1) ===
Index name : node_id
Index dtype: str
Shape      : (31, 1)


,base_demand
node_id,
10,-0.088232
11,-0.220781
12,-0.067373
13,0.832637
14,0.084998
15,-0.666576
16,-0.551775
17,0.368386
18,1.704304



=== pipe_s (scenario 1) ===
Index name : pipe_id
Index dtype: str
Shape      : (34, 3)


,length,diameter,roughness
pipe_id,,,
1,-1.267604,0.812742,0.105246
10,0.046092,0.344384,-1.382438
11,0.418696,0.383068,0.390962
12,3.553948,0.026051,-0.164037
13,-0.230315,-0.304426,1.143703
14,-0.643484,-0.345905,0.684074
15,-0.602079,-0.562329,0.473530
16,2.821046,-0.421935,0.062509
17,1.212534,-0.152240,0.598467


## Step 4 — Verify index alignment with GraphBuilder
Check that every key in `junction_idx` and `pipe_idx` exists in the sliced DataFrames,
and that the types match (string vs int mismatches cause silent `NaN` via `.loc`).

In [14]:
# Load n_bins_per_type from the saved metadata
processed_dir = os.path.join(config.data_dir, 'processed')
with open(os.path.join(processed_dir, 'metadata.json')) as f:
    meta = json.load(f)
n_bins_per_type = meta['n_bins_per_type']

gb = GraphBuilder(
    topology,
    n_bins_per_type,
    bidirectional=config.bidirectional_has_measure,
    virtual_node_mode=config.virtual_node_mode,
)

# --- Junction alignment ---
junc_idx_keys  = list(gb.junction_idx.keys())
junc_s_ids     = list(junc_s.index)
print("junction_idx key type :", type(junc_idx_keys[0]).__name__, " example:", repr(junc_idx_keys[0]))
print("junc_s index type     :", type(junc_s_ids[0]).__name__,   " example:", repr(junc_s_ids[0]))

missing_junc = [nid for nid in junc_idx_keys if nid not in junc_s.index]
print(f"\nJunction IDs in junction_idx but NOT in junc_s: {missing_junc}")

# --- Pipe alignment ---
pipe_idx_keys = list(gb.pipe_idx.keys())
pipe_s_ids    = list(pipe_s.index)
print("\npipe_idx key type :", type(pipe_idx_keys[0]).__name__, " example:", repr(pipe_idx_keys[0]))
print("pipe_s index type :", type(pipe_s_ids[0]).__name__,   " example:", repr(pipe_s_ids[0]))

missing_pipe = [pid for pid in pipe_idx_keys if pid not in pipe_s.index]
print(f"\nPipe IDs in pipe_idx but NOT in pipe_s: {missing_pipe}")

junction_idx key type : str  example: '2'
junc_s index type     : str  example: '10'

Junction IDs in junction_idx but NOT in junc_s: []

pipe_idx key type : str  example: '1'
pipe_s index type : str  example: '1'

Pipe IDs in pipe_idx but NOT in pipe_s: []


## Step 5 — Simulate `build_base`
Exact copy of the feature-building loops from `build_base`. Everything looks good, including the conversion from ids

In [15]:
# --- copied from graph_builder.py build_base ---

# junction: [base_demand]
junction_x = torch.zeros(len(gb.junction_idx), 1)
for nid, j in gb.junction_idx.items():
    junction_x[j, 0] = junc_s.loc[nid, 'base_demand']
    print(f"nid: {nid}, j:{j}")

# reservoir: zeros placeholder
reservoir_x = torch.zeros(len(gb.reservoir_idx), 1)

# pipe: [length, diameter, roughness]
pipe_x = torch.zeros(len(gb.pipe_idx), 3)
for pid, p in gb.pipe_idx.items():
    pipe_x[p, 0] = pipe_s.loc[pid, 'length']
    pipe_x[p, 1] = pipe_s.loc[pid, 'diameter']
    pipe_x[p, 2] = pipe_s.loc[pid, 'roughness']
    print(f"pid: {pid}, p:{p}")

print("=== Simulated junction_x ===")
print(f"Shape: {junction_x.shape}  NaN: {torch.isnan(junction_x).sum().item()}")
print(f"Min: {junction_x.min():.4f}  Max: {junction_x.max():.4f}")
print(junction_x.T)  # transposed for compact display

print("\n=== Simulated pipe_x ===")
print(f"Shape: {pipe_x.shape}  NaN: {torch.isnan(pipe_x).sum().item()}")
print(pipe_x)

print("\n=== Reservoir_x (zeros placeholder) ===")
print(reservoir_x.T)

nid: 2, j:0
nid: 3, j:1
nid: 4, j:2
nid: 5, j:3
nid: 6, j:4
nid: 7, j:5
nid: 8, j:6
nid: 9, j:7
nid: 10, j:8
nid: 11, j:9
nid: 12, j:10
nid: 13, j:11
nid: 14, j:12
nid: 15, j:13
nid: 16, j:14
nid: 17, j:15
nid: 18, j:16
nid: 19, j:17
nid: 20, j:18
nid: 21, j:19
nid: 22, j:20
nid: 23, j:21
nid: 24, j:22
nid: 25, j:23
nid: 26, j:24
nid: 27, j:25
nid: 28, j:26
nid: 29, j:27
nid: 30, j:28
nid: 31, j:29
nid: 32, j:30
pid: 1, p:0
pid: 2, p:1
pid: 3, p:2
pid: 4, p:3
pid: 5, p:4
pid: 6, p:5
pid: 7, p:6
pid: 8, p:7
pid: 9, p:8
pid: 10, p:9
pid: 11, p:10
pid: 12, p:11
pid: 13, p:12
pid: 14, p:13
pid: 15, p:14
pid: 16, p:15
pid: 17, p:16
pid: 18, p:17
pid: 19, p:18
pid: 20, p:19
pid: 21, p:20
pid: 22, p:21
pid: 23, p:22
pid: 24, p:23
pid: 25, p:24
pid: 26, p:25
pid: 27, p:26
pid: 28, p:27
pid: 29, p:28
pid: 30, p:29
pid: 31, p:30
pid: 32, p:31
pid: 33, p:32
pid: 34, p:33
=== Simulated junction_x ===
Shape: torch.Size([31, 1])  NaN: 0
Min: -1.0116  Max: 1.7043
tensor([[ 0.7335,  0.3415, -0.8916,  

## Step 6 — Compare against the saved base graph
Load the actual `base_{scenario}.pt` and check whether the feature tensors match what step 5 produced.

In [16]:
saved_base = torch.load(
    os.path.join(processed_dir, f'base_{scenario}.pt'),
    weights_only=False,
)

print("=== Saved base graph ===")
print(f"junction.x  shape: {saved_base['junction'].x.shape}   NaN: {torch.isnan(saved_base['junction'].x).sum().item()}")
print(f"reservoir.x shape: {saved_base['reservoir'].x.shape}   NaN: {torch.isnan(saved_base['reservoir'].x).sum().item()}")
print(f"pipe.x      shape: {saved_base['pipe'].x.shape}   NaN: {torch.isnan(saved_base['pipe'].x).sum().item()}")

print("\n=== junction.x match (simulated vs saved) ===")
junc_match = torch.allclose(junction_x, saved_base['junction'].x, atol=1e-5)
print(f"Matches: {junc_match}")
if not junc_match:
    diff = (junction_x - saved_base['junction'].x).abs()
    print(f"Max diff: {diff.max():.8f}")
    print("Simulated:\n", junction_x.T)
    print("Saved    :\n", saved_base['junction'].x.T)

print("\n=== pipe.x match (simulated vs saved) ===")
pipe_match = torch.allclose(pipe_x, saved_base['pipe'].x, atol=1e-5)
print(f"Matches: {pipe_match}")
if not pipe_match:
    diff = (pipe_x - saved_base['pipe'].x).abs()
    print(f"Max diff: {diff.max():.8f}")
    print("Simulated:\n", pipe_x)
    print("Saved    :\n", saved_base['pipe'].x)

print("\nSaved reservoir.x:")
print(saved_base['reservoir'].x.T)

=== Saved base graph ===
junction.x  shape: torch.Size([31, 1])   NaN: 0
reservoir.x shape: torch.Size([1, 1])   NaN: 0
pipe.x      shape: torch.Size([34, 3])   NaN: 0

=== junction.x match (simulated vs saved) ===
Matches: True

=== pipe.x match (simulated vs saved) ===
Matches: True

Saved reservoir.x:
tensor([[0.]])
